In [ ]:
from pathlib import Path
import pandas as pd
import re

# =========================
# 路徑：請改成你的實際路徑
# =========================
base_dir = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out")

reg_csv = base_dir / "appendix_bip.csv"
ox_csv  = base_dir / "appendix_oaxaca_bip.csv"

reg_tex = base_dir / "appendix_bip.tex"
ox_tex  = base_dir / "appendix_oaxaca_bip.tex"


# -------------------------
# Basic formatting helpers
# -------------------------
def latex_escape(text):
    if pd.isna(text):
        return ""
    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group(0)], text)


def stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def fmt_coef(x, p):
    if pd.isna(x):
        return ""
    return f"{x:.3f}{stars(p)}"


def fmt_se(se):
    if pd.isna(se):
        return ""
    return f"({se:.3f})"


def fmt_n(x):
    if pd.isna(x):
        return ""
    return f"{int(round(x)):,}"


def clean_df(df):
    df = df.copy()
    df["table"] = df["table"].astype(str).str.strip()
    df["col"] = pd.to_numeric(df["col"], errors="coerce").astype("Int64")
    df["model"] = df["model"].astype(str).str.strip()
    df["panel"] = df["panel"].astype(str).str.strip()
    df["row"] = df["row"].astype(str).str.strip()
    df["coef"] = pd.to_numeric(df["coef"], errors="coerce")
    df["se"] = pd.to_numeric(df["se"], errors="coerce")
    df["p"] = pd.to_numeric(df["p"], errors="coerce")
    df["N"] = pd.to_numeric(df["N"], errors="coerce")
    return df


# ============================================================
# 1. Regression table: Model 1 pooled, Model 2 BIP, WM
# ============================================================
def build_regression_table(input_csv, output_tex):
    df = clean_df(pd.read_csv(input_csv))

    row_order = [
        ("main", "BIP"),

        ("employment", "Employment reduction"),
        ("employment", "Job loss"),
        ("employment", "Key worker"),
        ("employment", "Self-employed"),

        ("financial", "Financial difficulties"),
        ("financial", "Financial better"),
        ("financial", "Financial worse"),

        ("housing", "Bedroom-per-person ratio"),
        ("housing", "Bedroom-per-person ratio squared"),
        ("housing", "Number of rooms"),
        ("housing", "Number of rooms squared"),

        ("health", "COVID symptoms"),
        ("health", "Respiratory condition"),
        ("health", "Cardiovascular condition"),
        ("health", "Endocrine condition"),
        ("health", "Arthritis"),
        ("health", "Other chronic condition"),

        ("household", "Single occupancy"),
        ("household", "Dependent children"),
        ("household", "Dependent children squared"),
        ("household", "Single occupancy x children"),
        ("household", "Single occupancy x children squared"),

        ("demographics", "Female"),
        ("demographics", "Age 16-29"),
        ("demographics", "Age 30-49"),
        ("demographics", "Age 50-69"),

        ("constant", "Constant"),
        ("footer", "Observations"),
    ]

    section_titles = {
        "main": "",
        "employment": "Employment and financial factors",
        "financial": "",
        "housing": "Housing",
        "health": "Health",
        "household": "Household composition",
        "demographics": "Demographics",
        "constant": "",
    }

    label_map = {
        "BIP": "BIP",
        "Employment reduction": r"\hspace{0.15cm}Employment reduction",
        "Job loss": r"\hspace{0.15cm}Job loss",
        "Key worker": r"\hspace{0.15cm}Key worker",
        "Self-employed": r"\hspace{0.15cm}Self-employed",
        "Financial difficulties": r"\hspace{0.15cm}Financial difficulties",
        "Financial better": r"\hspace{0.15cm}Financial status improved",
        "Financial worse": r"\hspace{0.15cm}Financial status worsened",
        "Bedroom-per-person ratio": r"\hspace{0.15cm}Bedroom-per-person ratio",
        "Bedroom-per-person ratio squared": r"\hspace{0.15cm}Bedroom-per-person ratio squared",
        "Number of rooms": r"\hspace{0.15cm}Number of rooms",
        "Number of rooms squared": r"\hspace{0.15cm}Number of rooms squared",
        "COVID symptoms": r"\hspace{0.15cm}COVID symptoms",
        "Respiratory condition": r"\hspace{0.15cm}Respiratory condition",
        "Cardiovascular condition": r"\hspace{0.15cm}Cardiovascular condition",
        "Endocrine condition": r"\hspace{0.15cm}Endocrine condition",
        "Arthritis": r"\hspace{0.15cm}Arthritis",
        "Other chronic condition": r"\hspace{0.15cm}Other chronic condition",
        "Single occupancy": r"\hspace{0.15cm}Single occupancy",
        "Dependent children": r"\hspace{0.15cm}Dependent children",
        "Dependent children squared": r"\hspace{0.15cm}Dependent children squared",
        "Single occupancy x children": r"\hspace{0.15cm}Single occupancy $\times$ children",
        "Single occupancy x children squared": r"\hspace{0.15cm}Single occupancy $\times$ children squared",
        "Female": r"\hspace{0.15cm}Female",
        "Age 16-29": r"\hspace{0.15cm}Age 16--29",
        "Age 30-49": r"\hspace{0.15cm}Age 30--49",
        "Age 50-69": r"\hspace{0.15cm}Age 50--69",
        "Constant": "Constant",
        "Observations": "Observations",
    }

    # Build cell dictionary
    cell = {}
    for _, r in df.iterrows():
        key = (r["panel"], r["row"], int(r["col"]))
        cell[key] = {
            "coef": fmt_coef(r["coef"], r["p"]),
            "se": fmt_se(r["se"]),
            "N": fmt_n(r["N"]),
        }

    def coef(panel, row, col):
        return cell.get((panel, row, col), {}).get("coef", "")

    def se(panel, row, col):
        return cell.get((panel, row, col), {}).get("se", "")

    def nval(panel, row, col):
        return cell.get((panel, row, col), {}).get("N", "")

    lines = []
    lines.append(r"\begin{landscape}")
    lines.append(r"\small")
    lines.append(r"\setlength{\tabcolsep}{5pt}")
    lines.append(r"\renewcommand{\arraystretch}{0.95}")
    lines.append(r"\begin{longtable}{@{}p{0.44\linewidth}>{\centering\arraybackslash}p{0.145\linewidth}>{\centering\arraybackslash}p{0.145\linewidth}>{\centering\arraybackslash}p{0.145\linewidth}@{}}")
    lines.append(r"\caption{Regression Models for Changes in Mental Well-Being}")
    lines.append(r"\label{tab:appendix3_regression}\\")
    lines.append(r"\toprule")
    lines.append(r" & (1) & (2) & (3) \\")
    lines.append(r" & \shortstack{Pooled\\model} & \shortstack{BIP\\subsample} & \shortstack{WM\\subsample} \\")
    lines.append(r"\midrule")
    lines.append(r"\endfirsthead")
    lines.append(r"\toprule")
    lines.append(r" & (1) & (2) & (3) \\")
    lines.append(r" & \shortstack{Pooled\\model} & \shortstack{BIP\\subsample} & \shortstack{WM\\subsample} \\")
    lines.append(r"\midrule")
    lines.append(r"\endhead")
    lines.append(r"\midrule")
    lines.append(r"\multicolumn{4}{r}{\textit{Continued on next page}} \\")
    lines.append(r"\endfoot")
    lines.append(r"\bottomrule")
    lines.append(r"\endlastfoot")

    last_section = None
    for panel, row in row_order:
        if panel != "footer":
            title = section_titles.get(panel, None)
            if title and panel != last_section:
                lines.append(r"\addlinespace[0.25em]")
                lines.append(r"\multicolumn{4}{l}{\textit{" + title + r"}} \\")
                last_section = panel

        label = label_map.get(row, latex_escape(row))

        if row == "Observations":
            lines.append(r"\addlinespace[0.25em]")
            lines.append(
                f"{label} & {nval(panel, row, 1)} & {nval(panel, row, 2)} & {nval(panel, row, 3)} \\\\"
            )
        else:
            lines.append(
                f"{label} & {coef(panel, row, 1)} & {coef(panel, row, 2)} & {coef(panel, row, 3)} \\\\"
            )
            lines.append(
                f" & {se(panel, row, 1)} & {se(panel, row, 2)} & {se(panel, row, 3)} \\\\"
            )

    lines.append(r"\end{longtable}")
    lines.append(r"")
    lines.append(r"\vspace{-0.8em}")
    lines.append(
    r"\noindent\parbox{0.95\linewidth}{\footnotesize "
    r"\textit{Notes:} The dependent variable is the individual change in standardized, seasonally adjusted and inverted GHQ Likert score. "
    r"The reported gap is defined as WM minus BIP. "
    r"Column (1) reports the pooled-price decomposition, while Column (2) evaluates the decomposition at WM prices. "
    r"Standard errors clustered at the primary sampling unit are reported in parentheses. "
    r"Detailed entries report the contribution of each variable to the composition and structural components of the Oaxaca--Blinder decomposition. "
    r"Employment no-change includes respondents not working before the pandemic. "
    r"Age 70+ is the omitted category. "
    r"* \(p<0.10\), ** \(p<0.05\), *** \(p<0.01\).}"
    )
    lines.append(r"\end{landscape}")

    output_tex.write_text("\n".join(lines), encoding="utf-8")
    print(f"Saved: {output_tex}")


# ============================================================
# 2. Oaxaca table
# ============================================================
def build_oaxaca_table(input_csv, output_tex):
    df = clean_df(pd.read_csv(input_csv))

    cell = {}
    for _, r in df.iterrows():
        key = (r["panel"], r["row"], int(r["col"]))
        cell[key] = {
            "coef": fmt_coef(r["coef"], r["p"]),
            "se": fmt_se(r["se"]),
            "N": fmt_n(r["N"]),
        }

    def coef(panel, row, col=1):
        return cell.get((panel, row, col), {}).get("coef", "")

    def se(panel, row, col=1):
        return cell.get((panel, row, col), {}).get("se", "")

    def nval(panel, row, col=1):
        return cell.get((panel, row, col), {}).get("N", "")

    def label(row):
        label_map = {
            "Group gap": "Group gap",
            "Total composition effect": "Total composition effect",
            "Total structural effect": "Total structural effect",

            "Employment reduction": r"\hspace{0.15cm}Employment reduction",
            "Job loss": r"\hspace{0.15cm}Job loss",
            "Key worker": r"\hspace{0.15cm}Key worker",
            "Self-employed": r"\hspace{0.15cm}Self-employed",

            "Financial difficulties": r"\hspace{0.15cm}Financial difficulties",
            "Financial status improved": r"\hspace{0.15cm}Financial status improved",
            "Financial status worsened": r"\hspace{0.15cm}Financial status worsened",

            "Bedroom-per-person ratio": r"\hspace{0.15cm}Bedroom-per-person ratio",
            "Bedroom-per-person ratio squared": r"\hspace{0.15cm}Bedroom-per-person ratio squared",
            "Number of rooms": r"\hspace{0.15cm}Number of rooms",
            "Number of rooms squared": r"\hspace{0.15cm}Number of rooms squared",

            "COVID symptoms": r"\hspace{0.15cm}COVID symptoms",
            "Respiratory condition": r"\hspace{0.15cm}Respiratory condition",
            "Cardiovascular condition": r"\hspace{0.15cm}Cardiovascular condition",
            "Endocrine condition": r"\hspace{0.15cm}Endocrine condition",
            "Arthritis": r"\hspace{0.15cm}Arthritis",
            "Other chronic condition": r"\hspace{0.15cm}Other chronic condition",

            "Single occupancy": r"\hspace{0.15cm}Single occupancy",
            "Dependent children": r"\hspace{0.15cm}Dependent children",
            "Dependent children squared": r"\hspace{0.15cm}Dependent children squared",
            "Single occupancy x children": r"\hspace{0.15cm}Single occupancy $\times$ children",
            "Single occupancy x children squared": r"\hspace{0.15cm}Single occupancy $\times$ children squared",

            "Female": r"\hspace{0.15cm}Female",
            "Age 16-29": r"\hspace{0.15cm}Age 16--29",
            "Age 30-49": r"\hspace{0.15cm}Age 30--49",
            "Age 50-69": r"\hspace{0.15cm}Age 50--69",

            "Constant": r"\hspace{0.15cm}Constant",
            "Observations": "Observations",
        }
        return label_map.get(row, latex_escape(row))

    rows = [
        ("row", "overall", "Group gap"),

        ("section", "Composition effect attributable to:", ""),
        ("subsection", "Employment factors", ""),
        ("row", "explained_employment", "Employment reduction"),
        ("row", "explained_employment", "Job loss"),
        ("row", "explained_employment", "Key worker"),
        ("row", "explained_employment", "Self-employed"),

        ("subsection", "Financial factors", ""),
        ("row", "explained_financial", "Financial difficulties"),
        ("row", "explained_financial", "Financial status improved"),
        ("row", "explained_financial", "Financial status worsened"),

        ("subsection", "Housing", ""),
        ("row", "explained_housing", "Bedroom-per-person ratio"),
        ("row", "explained_housing", "Bedroom-per-person ratio squared"),
        ("row", "explained_housing", "Number of rooms"),
        ("row", "explained_housing", "Number of rooms squared"),

        ("subsection", "Health", ""),
        ("row", "explained_health", "COVID symptoms"),
        ("row", "explained_health", "Respiratory condition"),
        ("row", "explained_health", "Cardiovascular condition"),
        ("row", "explained_health", "Endocrine condition"),
        ("row", "explained_health", "Arthritis"),
        ("row", "explained_health", "Other chronic condition"),

        ("subsection", "Household composition", ""),
        ("row", "explained_household", "Single occupancy"),
        ("row", "explained_household", "Dependent children"),
        ("row", "explained_household", "Dependent children squared"),
        ("row", "explained_household", "Single occupancy x children"),
        ("row", "explained_household", "Single occupancy x children squared"),

        ("subsection", "Demographics", ""),
        ("row", "explained_demographics", "Female"),
        ("row", "explained_demographics", "Age 16-29"),
        ("row", "explained_demographics", "Age 30-49"),
        ("row", "explained_demographics", "Age 50-69"),

        ("row", "overall", "Total composition effect"),

        ("section", "Structural effect attributable to:", ""),
        ("subsection", "Employment factors", ""),
        ("row", "unexplained_employment", "Employment reduction"),
        ("row", "unexplained_employment", "Job loss"),
        ("row", "unexplained_employment", "Key worker"),
        ("row", "unexplained_employment", "Self-employed"),

        ("subsection", "Financial factors", ""),
        ("row", "unexplained_financial", "Financial difficulties"),
        ("row", "unexplained_financial", "Financial status improved"),
        ("row", "unexplained_financial", "Financial status worsened"),

        ("subsection", "Housing", ""),
        ("row", "unexplained_housing", "Bedroom-per-person ratio"),
        ("row", "unexplained_housing", "Bedroom-per-person ratio squared"),
        ("row", "unexplained_housing", "Number of rooms"),
        ("row", "unexplained_housing", "Number of rooms squared"),

        ("subsection", "Health", ""),
        ("row", "unexplained_health", "COVID symptoms"),
        ("row", "unexplained_health", "Respiratory condition"),
        ("row", "unexplained_health", "Cardiovascular condition"),
        ("row", "unexplained_health", "Endocrine condition"),
        ("row", "unexplained_health", "Arthritis"),
        ("row", "unexplained_health", "Other chronic condition"),

        ("subsection", "Household composition", ""),
        ("row", "unexplained_household", "Single occupancy"),
        ("row", "unexplained_household", "Dependent children"),
        ("row", "unexplained_household", "Dependent children squared"),
        ("row", "unexplained_household", "Single occupancy x children"),
        ("row", "unexplained_household", "Single occupancy x children squared"),

        ("subsection", "Demographics", ""),
        ("row", "unexplained_demographics", "Female"),
        ("row", "unexplained_demographics", "Age 16-29"),
        ("row", "unexplained_demographics", "Age 30-49"),
        ("row", "unexplained_demographics", "Age 50-69"),

        ("row", "unexplained_constant", "Constant"),
        ("row", "overall", "Total structural effect"),

        ("row", "footer", "Observations"),
    ]

    lines = []
    lines.append(r"\begin{landscape}")
    lines.append(r"\small")
    lines.append(r"\setlength{\tabcolsep}{1.5pt}")
    lines.append(r"\renewcommand{\arraystretch}{0.95}")
    lines.append(r"\begin{longtable}{@{}p{0.54\linewidth}>{\centering\arraybackslash}p{0.18\linewidth}>{\centering\arraybackslash}p{0.18\linewidth}@{}}")
    lines.append(r"\caption{Detailed Oaxaca--Blinder Decomposition of the WM--BIP Gap}")
    lines.append(r"\label{tab:appendix3_oaxaca}\\")
    lines.append(r"\toprule")
    lines.append(r" & (1) & (2) \\")
    lines.append(r" & Pooled price & WM price \\")
    lines.append(r"\midrule")
    lines.append(r"\endfirsthead")
    lines.append(r"\toprule")
    lines.append(r" & (1) & (2) \\")
    lines.append(r" & Pooled price & WM price \\")
    lines.append(r"\midrule")
    lines.append(r"\endhead")
    lines.append(r"\midrule")
    lines.append(r"\multicolumn{3}{r}{\textit{Continued on next page}} \\")
    lines.append(r"\endfoot")
    lines.append(r"\bottomrule")
    lines.append(r"\endlastfoot")

    for kind, panel, row in rows:
        if kind == "section":
            lines.append(r"\addlinespace[0.3em]")
            lines.append(r"\multicolumn{3}{l}{\textit{" + panel + r"}} \\")
        elif kind == "subsection":
            lines.append(r"\addlinespace[0.2em]")
            lines.append(r"\multicolumn{3}{l}{\textit{" + panel + r"}} \\")
        else:
            if row == "Observations":
                lines.append(r"\addlinespace[0.25em]")
                lines.append(f"{label(row)} & {nval(panel, row, 1)} & {nval(panel, row, 2)} \\\\")
            else:
                lines.append(f"{label(row)} & {coef(panel, row, 1)} & {coef(panel, row, 2)} \\\\")
                lines.append(f" & {se(panel, row, 1)} & {se(panel, row, 2)} \\\\")

    lines.append(r"\end{longtable}")
    lines.append(r"")
    lines.append(r"\vspace{-0.8em}")
    lines.append(
        r"\noindent\parbox{0.95\linewidth}{\footnotesize "
        r"\textit{Notes:} The dependent variable is the individual change in standardized, seasonally adjusted and inverted GHQ Likert score. "
        r"Column (1) reports the pooled regression including a BIP indicator. "
        r"Columns (2) and (3) report subgroup regressions for BIP and WM respondents, respectively. "
        r"All models use CA survey weights. "
        r"Standard errors are calculated using the survey design and reported in parentheses. "
        r"Employment no-change includes respondents not working before the pandemic. "
        r"Age 70+ is the omitted category. "
        r"* \(p<0.10\), ** \(p<0.05\), *** \(p<0.01\).}"
)
    lines.append(r"\end{landscape}")

    output_tex.write_text("\n".join(lines), encoding="utf-8")
    print(f"Saved: {output_tex}")

# Run
build_regression_table(reg_csv, reg_tex)
build_oaxaca_table(ox_csv, ox_tex)

Saved: /Users/lishixue/Documents/Master thesis/Statafile/out/appendix3_regression_bip.tex
Saved: /Users/lishixue/Documents/Master thesis/Statafile/out/appendix3_oaxaca_bip.tex
